# 01 · B0 text classifier (TF-IDF char n-grams + logistic regression)

Builds the synthetic multilingual dataset → trains **B0** → evaluates on the held-out test families → packages the artifact for the backend.

> **All data in this notebook is SYNTHETIC** (template-generated). Metrics are a regression/ablation signal on unseen template families — **not** an estimate of real-world accuracy. Hindi/Marathi text needs native-speaker review.

**Settings:** CPU is enough (≈1–2 min). **Internet = ON** only so the first cell can clone the project repo from GitHub (not needed if you attach a Dataset containing the repo's `ai/` folder). Run all cells top to bottom.

In [ ]:
# --- 0. Locate the CivicConnect repo and set up a work directory (works on Kaggle and locally) ---------------
import importlib.util
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

ON_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))

def find_repo() -> Path:
    """Return a writable directory that contains ai/inference/schemas.py."""
    marker = Path("ai/inference/schemas.py")
    if os.environ.get("CIVIC_REPO"):                                   # explicit override
        return Path(os.environ["CIVIC_REPO"])
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:                                    # running from inside the repo
        if (p / marker).exists():
            return p
    if ON_KAGGLE:                                                      # attached Kaggle Dataset containing the repo or just ai/
        for hit in sorted(Path("/kaggle/input").glob("**/ai/inference/schemas.py"))[:1]:
            src = hit.parents[2]                                       # .../ai/inference/schemas.py -> parent of ai/
            dst = Path("/kaggle/working/repo")
            if not dst.exists():
                shutil.copytree(src, dst)                              # /kaggle/input is read-only
            return dst
        url = os.environ.get("CIVIC_GIT_URL", "https://github.com/tanujb03/CivicConnect_v2")                          # default: clone the project repo's `tanuj` branch (needs Internet ON); override with CIVIC_GIT_URL / CIVIC_GIT_REF
        if url:
            subprocess.run(["git", "clone", "--depth", "1", "--branch", os.environ.get("CIVIC_GIT_REF", "tanuj"), url, "/kaggle/working/repo"], check=True)
            return Path("/kaggle/working/repo")
    raise SystemExit("Cannot find the repo. On Kaggle: turn Internet ON (the notebook clones the project from GitHub) or add a Dataset containing the repo's `ai/` folder (see ai/training/README.md).")

REPO = find_repo()
sys.path.insert(0, str(REPO))
WORK = Path(os.environ.get("CIVIC_WORKDIR") or ("/kaggle/working" if ON_KAGGLE else tempfile.mkdtemp(prefix="civic_work_")))
WORK.mkdir(parents=True, exist_ok=True)

_MODULE = {"scikit-learn": "sklearn", "sentence-transformers": "sentence_transformers"}

def ensure(*pkgs):
    """pip install only what is missing (Kaggle images already ship numpy/scipy/scikit-learn/pydantic/httpx)."""
    missing = [p for p in pkgs if importlib.util.find_spec(_MODULE.get(p, p)) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

ensure("numpy", "scipy", "scikit-learn", "pydantic", "httpx")
print(f"repo : {REPO}\nwork : {WORK}\npython {sys.version.split()[0]} | kaggle={ON_KAGGLE}")

## 1 · Build the synthetic dataset
Splits are by **template family and place pool**. The cell compares hashes with the committed reference manifest (`ai/evaluation/datasets/manifest.v1.json`); a mismatch is reported but not fatal (can happen across Python minor versions).

In [ ]:
from ai.training.src import build_dataset

DATA = WORK / "synthetic_v1"
build_dataset.main(["--out", str(DATA), "--check-manifest", str(REPO / "ai/evaluation/datasets/manifest.v1.json")])

## 2 · Train B0
Model selection (C) and temperature calibration use the *validation* families; the shipped model is refit on train+val. The script ends with a **parity check**: the numpy artifact must reproduce scikit-learn's probabilities.

In [ ]:
from ai.training.src import train_text_classifier

ART_ROOT = WORK / "artifacts" / "civic_text_b0"
ART_DIR = train_text_classifier.run(DATA, ART_ROOT)

## 3 · Evaluate on the held-out test set (unseen template families + places)
Uses the **production** `IntakeService` fallback path, not notebook code.

In [ ]:
import json

from ai.evaluation import run_eval

REPORTS = WORK / "reports"
rc = run_eval.main(["--task", "intake", "--system", "local", "--artifact", str(ART_DIR), "--eval-dir", str(REPO / "ai/evaluation/datasets"),
                    "--out", str(REPORTS), "--name", "intake_b0_local"])
assert rc == 0
rep = json.loads((REPORTS / "intake_b0_local.json").read_text(encoding="utf-8"))["results"]
print(f"\nN={rep['n']}  category acc={rep['category_accuracy']}  subcategory acc={rep['subcategory_accuracy']} (95% CI {rep['subcategory_accuracy_ci95']})")
print(f"macro-F1={rep['subcategory_macro_f1']}  ECE={rep['calibration_ece_category']}  coverage@0.55={rep['coverage_at_threshold']}  acc when accepted={rep['category_accuracy_when_accepted']}")
for lang, v in rep["by_language"].items():
    print(f"  {lang:8s} n={v['n']:3d}  subcategory acc={v['subcategory_accuracy']}")

### 3b · Training-data policy, and the seeded demo city (design §58)
**What trains B0:** only the synthetic template dataset built in cell 1 (train families; validation families for model selection/calibration). That is deliberate: synthetic data is the *training and regression* source; **real data (notebook 05) is evaluation-only** until licences and the label mapping are verified.

**The demo city is not training data.** It is a seeded synthetic city (wards, 560 cases, planted duplicates / recurring sites / hotspots / incidents) used to demo and to check the pipeline end to end. Its report texts come from the same generator, and only the cases marked `text_split_pool == test` are held-out families — the cell below scores those separately from the rest.

In [ ]:
from ai.evaluation import eval_demo_city

rc = eval_demo_city.main(["--artifact", str(ART_DIR), "--out", str(REPORTS), "--name", "demo_city_synthetic"])
assert rc == 0
city = json.loads((REPORTS / "demo_city_synthetic.json").read_text(encoding="utf-8"))["results"]
print(json.dumps(city, indent=1, ensure_ascii=False)[:3500])

## 4 · (Optional) 5-fold family-grouped cross-validation
Each fold holds out a different template family per class/language, giving a variance estimate. Set `RUN_CV = False` to skip (≈1 min).

In [ ]:
import numpy as np
from ai.evaluation import eval_intake
from ai.inference.intake.service import IntakeService
from ai.inference.local.text_classifier import LocalTextClassifier
from ai.training.src.io_utils import read_jsonl

RUN_CV = os.environ.get("CIVIC_SKIP_CV") != "1"
if RUN_CV:
    scores = []
    for k in range(5):
        d = WORK / "cv" / f"fold{k}"
        build_dataset.build(d, seed=42, test_fold=k, n_train=10, n_val=6, n_test=2, pairs_train=100, pairs_val=50, pairs_test=50)
        art = train_text_classifier.run(d, WORK / "cv" / f"art{k}")
        res = eval_intake.evaluate(IntakeService(None, LocalTextClassifier.load(art)), read_jsonl(d / "test.jsonl"))
        scores.append((res["category_accuracy"], res["subcategory_accuracy"]))
        print(f"fold {k}: category={res['category_accuracy']}  subcategory={res['subcategory_accuracy']}")
    a = np.array(scores)
    print(f"\nmean±std  category {a[:,0].mean():.3f}±{a[:,0].std():.3f}   subcategory {a[:,1].mean():.3f}±{a[:,1].std():.3f}")

## 5 · Smoke-test the artifact through the production inference adapter

In [ ]:
from ai.inference.schemas import IntakeRequest
from ai.inference.service import AIService

ai = AIService.from_env({"AI_LOCAL_CLASSIFIER_PATH": str(ART_DIR)})
for text in ["There is a big pothole near the school gate", "सड़क पर बड़ा गड्ढा है", "रस्त्यावर कचरा ओसंडून वाहत आहे", "paani ki pipe leak ho rahi hai"]:
    r = ai.analyze_intake(IntakeRequest(text=text))
    p = r.proposal
    print(f"{text[:45]:47s} -> {p.category}/{p.subcategory}  sev={p.severity}  dept={p.suggested_department}  conf={r.confidence:.2f}  lang={p.language}  src={r.ai_metadata.source}")
print("\nstatus:", ai.status()["local_classifier"])

## 6 · Package the hand-off bundle
Download the zip from the notebook **Output** tab and unzip it into `ai/artifacts/civic_text_b0/` in the repo (set `AI_LOCAL_CLASSIFIER_PATH` to the version folder). The dataset itself is not included — it is regenerated from the seed.

In [ ]:
import hashlib
import zipfile

BUNDLE = WORK / f"civic_b0_handoff_{ART_DIR.name.replace('+', '_')}.zip"
with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(ART_DIR.iterdir()):
        z.write(f, f"civic_text_b0/{ART_DIR.name}/{f.name}")
    for f in sorted(REPORTS.iterdir()):
        z.write(f, f"reports/{f.name}")
    z.write(DATA / "DATASET_MANIFEST.json", "datasets/synthetic_v1/DATASET_MANIFEST.json")
print("bundle:", BUNDLE, f"({BUNDLE.stat().st_size/1e6:.2f} MB)")
print("weights.npz sha256:", hashlib.sha256((ART_DIR / "weights.npz").read_bytes()).hexdigest())
print("bundle      sha256:", hashlib.sha256(BUNDLE.read_bytes()).hexdigest())